# Script creating new .inp files for Loadest

Input: 
1. Original csv file from the Environment Canada database
2. Hydrometric flow data (m3/s) until 2016 from: https://wateroffice.ec.gc.ca/mainmenu/historical_data_index_e.html

Output: 
1. new calib.inp files for all EC variables

Comments:

1. Lack yellowknife data
2. Lack hydrometric data for 2017-2018
3. Lack TOC data
4. How to add Hg data from NWT output?

A.L.Soerensen June 2019

In [29]:
#Libraries
import pandas as pd
import numpy as np
from datetime import datetime
from datetime import timedelta

######################################
# Data that might need to be modified
######################################
#
# Start date & End date
startdate = '2000-01-01 00:00:00'
enddate = '2019-01-01 00:00:00'

######################################
# Rivers to isolate
# NW10ED0002 - Liard River mouth 
# NW10LA0003 - Mackenzie River - Arctic Red River # Mackenzie River @ Tsiigehtchic
# NW10MC0001 - Peel River 
EC_loc = ['NW10MC0001','NW10LA0003','NW10ED0002']

# Short name of rivers used in .inp file (needs to match rivers above)
River_short=['Peel River','Arctic Red River','Liard River']#,'Yellowknife River']

# Station to isolate for hydrological information
#            Peel     Arctic Red   Liard     Yellow
file_name = ['10MC002','10LA002','10ED002']#,'07SB003']

########################################
# Constituent lists (names and unit)
#
# List of constituent to make output files for
EC_list = ['DOC_mgL','POC_mgL','RES_NON-FIL_mgL','DS_mgL']

# Lists of names and units in original file
EC_VAR = ["CARBON DISSOLVED ORGANIC","CARBON PARTICULATE ORGANIC",
            "RESIDUE NONFILTRABLE", "SULPHATE DISSOLVED"]

EC_UNIT = ["MG/L","MG/L","MG/L","MG/L"]

################################################
# File paths
#
# path for NWT monitoring data
EC_name='/Users/staffan/Dropbox/Mackenzie Hg project/Supporting data/Environment Canada water quality data/'

# Path for hydrometric water flow file
Flow_name = '/Users/staffan/Dropbox/Mackenzie Hg project/Supporting data/Hydrological_data/Daily_'

# Base path to Loadest folder
Loadest = '/Users/staffan/Desktop/Loadest/EC/'

######################################################
# No need to change
#
# Path for original .inp file used as a template for all the new files
path = Loadest+'calib.inp'

# Paths for temporary files
Temp_name = Loadest+'txt_temp'

# Paths for final output files
Out_name = Loadest+'new_files'

In [34]:
######## READ DATA ###################
# Read the two original EC files
fname1=EC_name+'Water-Qual-Eau-Mackenzie-2000-p.xlsx'
df1 = pd.read_excel(fname1, sheet_name='Water-Qual-Eau-Mackenzie-2000-p')
df1 = df1.drop(['FLAG_MARQUEUR','VMV_CODE','SDL_LDE','MDL_LDM','STATUS_STATUT','VARIABLE_FR'], axis=1)
df1 = df1.loc[lambda AA: AA.DATE_TIME_HEURE > startdate, :]
df1 = df1.loc[lambda AA: AA.DATE_TIME_HEURE < enddate, :]

fname2=EC_name+'Water-Qual-Eau-Peace-Athabasca-2000-present.xlsx'
df2 = pd.read_excel(fname2, sheet_name='Water-Qual-Eau-Peace-Athabasca')
df2 = df2.drop(['FLAG_MARQUEUR','VMV_CODE','SDL_LDE','MDL_LDM','STATUS_STATUT','VARIABLE_FR'], axis=1)
df2 = df2.loc[lambda AA: AA.DATE_TIME_HEURE > startdate, :]
df2 = df2.loc[lambda AA: AA.DATE_TIME_HEURE < enddate, :]

# append the two files
df1 = df1.append(df2)
####################################
# Extract relevant stations
df3 = df1.set_index('SITE_NO')
df3 = df3.loc[EC_loc,:]
df4 = df3.reset_index()

# Add time variables
df4['year'] = pd.DatetimeIndex(df4['DATE_TIME_HEURE']).year
df4['month'] = pd.DatetimeIndex(df4['DATE_TIME_HEURE']).month 
df4['day'] = pd.DatetimeIndex(df4['DATE_TIME_HEURE']).day
df4 = df4.set_index(['SITE_NO','VARIABLE','year','month','day'])

# Create list of variables that can be looped over to set up columns for each variable
df4m = df4.groupby(level = ['SITE_NO','VARIABLE','year','month','day']).mean()
df4m = df4m.reset_index()
df4m = df4m.set_index('VARIABLE')

# Loop over variables and make them into seperate columns 
leng = 0

for id in EC_VAR:
    DOC = df4m.loc[[id],:]
    name = id+'_'+EC_UNIT[leng]
    DOC = DOC.reset_index()
    DOC = DOC.rename(index=str, columns={"VALUE_VALEUR":name}).drop(['VARIABLE'], axis=1)
    leng = leng+1

    if id == EC_VAR[0]:
        concat_m = DOC

    if id > EC_VAR[0]:
        DOC_m = DOC
        concat_m=pd.merge(concat_m,DOC_m,how='left',left_on=['SITE_NO','year','month','day'],right_on=['SITE_NO','year','month','day'])

# Add river names based on ID
for id in range(len(EC_loc)):
    concat_m.loc[concat_m.SITE_NO == EC_loc[id], 'Location'] = River_short[id]
    
# Add time variable to dataset
concat_m['time']=1200

# Reformat Date variable to correct format and merge new format to dataframe
df0 = concat_m.loc[:,('year','month','day')]

df0['date1']=pd.to_datetime(df0)

df0['date'] = df0.date1.dt.strftime("%Y%m%d")

df0 = df0.drop(['year','month','day','date1'], axis=1)

concat_m = pd.merge(concat_m,df0,left_index=True, right_index=True)

# Rename constituents
for id in range (len(EC_list)):
    concat_m = concat_m.rename(index=str, columns={(EC_VAR[id]+'_'+EC_UNIT[id]):EC_list[id]})

concat_m = concat_m.drop(['year','month','day'], axis=1)

concat_m.head()

,SITE_NO,DOC_mgL,POC_mgL,RES_NON-FIL_mgL,DS_mgL,Location,time,date
0,NW10ED0002,2.9,0.210,NaN,50.7,Liard River,1200,20000126
1,NW10ED0002,2.9,0.623,NaN,46.0,Liard River,1200,20000327
2,NW10ED0002,11.1,7.080,436.0,2.8,Liard River,1200,20010531
3,NW10ED0002,7.1,8.520,819.0,24.0,Liard River,1200,20010612
4,NW10ED0002,2.6,36.600,53.0,57.8,Liard River,1200,20020327


In [31]:
########################################
# Import hydrometric flow data and merge 
########################################

# Create empty dataframe to store data for all rivers
Q_all = pd.DataFrame()

# Read in relevant flow data and combine it
for X in file_name:
    Qname = Flow_name+X+'.csv'
    Q = pd.read_csv(Qname, skiprows=1)
    Q = Q.loc[lambda AA: AA.Date > startdate, :]
    Q = Q.loc[lambda BB: BB.PARAM == 1,:]
    Q['Q'] = Q['Value']*35.3
    Q = Q.drop(['PARAM','SYM','Value'],axis=1)
    Q_all = Q_all.append(Q)

# Add river names based on ID
Q_all = Q_all.rename(index=str, columns={" ID":"SITE"})
n = len(file_name)

for id in range(n):
    Q_all.loc[Q_all.SITE == file_name[id], 'Location'] = River_short[id]
    
# Reformat date
Q_all = Q_all.reset_index()
Q_all["Date"]=pd.to_datetime(Q_all.Date)
Q_all['date'] = Q_all.Date.dt.strftime("%Y%m%d")
Q_all = Q_all.drop(['Date','SITE','index'], axis=1)

######################################################
# Create empty dataframe to store data for all rivers
New_all = pd.DataFrame()

i=0
# Read in relevant flow data and combine it
for X in file_name:
    fname1='/Users/staffan/Dropbox/Mackenzie Hg project/Supporting data/Hydrological_data/2018_Daily_'+X+'.xlsx'
    New = pd.read_excel(fname1, sheet_name='Sheet1')
    New['Q']=New['Mean (m3/s)']*35.3
    #subtract a year (2019-1) as the year is wrong in the input file
    New['Day (m-d)']=New['Day (m-d)']- timedelta(days=365)
    New['date'] = New['Day (m-d)'].dt.strftime("%Y%m%d")
    New = New.drop(['Max (m3/s)','Min (m3/s)','Median (m3/s)','Upper Quartile (m3/s)','Lower Quartile (m3/s)',
                    'Mean (m3/s)','Day (m-d)'], axis=1)
    New['Location']=River_short[i]
    New = New[['Q','Location','date']]
    New_all = New_all.append(New)
    i=i+1
    
Q_all = Q_all.append(New_all)

# Merge constituent data with hydrological flows
dfQ =pd.merge(concat_m,Q_all,how='left',left_on=['Location','date'],right_on=['Location','date'])
dfQ = dfQ.dropna(subset=['Q'])

In [32]:
############################################
# Create .inp files based on merged constituent and hydrological flow file
##############################################

i=0
# Loop over the different variables that we want to create input files for
for idd in River_short:
    for id in EC_list:

        dfV = dfQ.loc[lambda BB: BB.Location == idd,:]
        
        # Extract column including specified constituent
        dfV = dfV.loc[:,('date','time','Q',id)]

        # Save excel input as a textfile
        Oname = Temp_name+'/calib_temp_EC_'+id+'.txt'
        dfV.to_csv(Oname, sep='\t', index=False, header=False)

        # Import text file
        Tname = Temp_name+'/calib_temp_EC_'+id+'.txt'
        calib_file = open(Tname,'r')
        calib = calib_file.read()

        ###########################################
        # Import original calib.inp file
        days_file = open(path,'r')
        days = days_file.read()

        ############################################
        # Combine the header with the datafile
        days1 = days[1:106]
        days01 = idd
        days02 = days[116:121]
        days2 = id
        days3 = days[123:309] #change the last number if there is problems ligning up the first date
        days4 = days1+days01+days02+days2+days3+calib

        ############################################
        # Write new calib.inp file
        new_path = Out_name+'/calib_EC_'+idd+"_"+id+'.inp'
        new_days = open(new_path,'w')
        
        new_days.write(days4)
        days_file.close()
        new_days.close()

    i=i+1

In [33]:
#print(id)
print(days4)

#####################################################################
#
#  LOADEST Calibration File
#
#  Liard Riveriver DS_mgL Marseilles, Illinois (Helsel & Hirsch, 2002)
#
#  Note: Sample dates (CDATE) were extracted from the decimal times
#        given by Helsel and Hirsch (2002).  Sample times (CTIME) are
20000126	1200	18108.899999999998	50.7
20000327	1200	13308.099999999999	46.0
20010531	1200	170146.0	2.8
20010612	1200	370649.99999999994	24.0
20020327	1200	13872.9	57.8
20020613	1200	339939.0	27.0
20020924	1200	94956.99999999999	39.4
20021007	1200	82955.0	41.2
20030204	1200	22909.699999999997	52.7
20030603	1200	166969.0	28.4
20030710	1200	234744.99999999997	33.2
20030918	1200	87544.0	44.0
20031030	1200	66717.0	42.4
20040217	1200	16838.1	49.5
20040521	1200	168734.0	23.7
20040808	1200	106958.99999999999	39.6
20041019	1200	74483.0	42.7
20050207	1200	17650.0	48.2
20050324	1200	12990.4	45.8
20050516	1200	217094.99999999997	25.0
20050622	1200	231567.99999999997	31.4
20050802	1200	16238